# Classification metrics from scratch

This notebook implements the classification metrics of notes.md §1 and §2 with numpy
only: the confusion matrix, precision, recall and F1, the ROC curve with AUC computed
two independent ways, and the precision-recall (PR) curve with average precision (AP).
sklearn is used only to check the results.

It ends with the imbalance experiment from notes.md §2 ("Why ROC hides the cost of class
imbalance"): with the score distributions held fixed, adding 100 times more negatives
leaves AUC unchanged while precision and AP fall sharply.

Run: `uv run jupyter execute ml/05_evaluation/metrics_scratch.ipynb`

In [1]:
import numpy as np

np.random.seed(42)

## 1. Confusion matrix, precision, recall, F1

The four functions below count TP, FP, FN and TN for binary labels and form the three
ratios from notes.md §1. To check them, the cell rebuilds the cancer-screening example
from §1: 1,000 people, 10 with cancer, TP = 8, FN = 2, FP = 90, TN = 900. Expect
accuracy 0.908, precision 8/98 = 0.082, recall 0.8, and an F1 of about 0.15, pulled
toward the small precision.

In [2]:
def confusion_matrix(y_true, y_pred):
    """Return (tp, fp, fn, tn) for binary labels in {0, 1}."""
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int(np.sum(y_true & y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    return tp, fp, fn, tn


def precision_score(y_true, y_pred):
    tp, fp, _, _ = confusion_matrix(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall_score(y_true, y_pred):
    tp, _, fn, _ = confusion_matrix(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_score(y_true, y_pred):
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


# Cancer-screening example from notes.md section 1.
y_true = np.r_[np.ones(10, int), np.zeros(990, int)]
y_pred = np.r_[np.ones(8, int), np.zeros(2, int),      # positives: 8 caught, 2 missed
               np.ones(90, int), np.zeros(900, int)]   # negatives: 90 false alarms
tp, fp, fn, tn = confusion_matrix(y_true, y_pred)
acc = (tp + tn) / len(y_true)
p, r, f1 = precision_score(y_true, y_pred), recall_score(y_true, y_pred), f1_score(y_true, y_pred)
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"accuracy={acc:.3f}  precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}")
print(f"'predict nobody has cancer' accuracy = {990/1000:.3f} (higher, and useless)")
assert (tp, fp, fn, tn) == (8, 90, 2, 900)
assert abs(acc - 0.908) < 1e-12 and abs(p - 8 / 98) < 1e-12 and abs(r - 0.8) < 1e-12

TP=8 FP=90 FN=2 TN=900
accuracy=0.908  precision=0.082  recall=0.800  F1=0.148
'predict nobody has cancer' accuracy = 0.990 (higher, and useless)


## 2. ROC curve and AUC, two independent ways

`roc_curve` sorts the examples by score and lowers the threshold past one distinct score
at a time: each positive admitted moves TPR up, each negative moves FPR right.
`auc_trapezoid` integrates that staircase. `auc_rank_statistic` instead computes the
fraction of positive-negative pairs ranked correctly, via the Mann-Whitney rank-sum
formula, which notes.md §2 proves is the same number.

The cell runs both on the 6-example table from notes.md §2 (labels 1,1,0,1,0,0 in score
order) and also calls sklearn. Expect the staircase up, up, right, up, right, right, and
all three AUC values equal to 8/9 = 0.889.

In [3]:
def roc_curve(y_true, scores):
    """Threshold sweep. Returns (fpr, tpr, thresholds), thresholds descending.

    Sort by score descending; lowering the threshold past each distinct score
    admits one more batch of predictions as positive. Positives move TPR up,
    negatives move FPR right.
    """
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="stable")
    y_sorted = y_true[order]
    s_sorted = scores[order]

    # cumulative TP/FP as threshold sweeps down
    tps = np.cumsum(y_sorted)
    fps = np.cumsum(1 - y_sorted)

    # keep only the last index of each run of tied scores
    distinct = np.where(np.diff(s_sorted))[0]
    idx = np.r_[distinct, y_sorted.size - 1]

    tps, fps = tps[idx], fps[idx]
    n_pos, n_neg = tps[-1], fps[-1]
    tpr = np.r_[0.0, tps / max(n_pos, 1)]
    fpr = np.r_[0.0, fps / max(n_neg, 1)]
    thresholds = np.r_[np.inf, s_sorted[idx]]
    return fpr, tpr, thresholds


def auc_trapezoid(fpr, tpr):
    """Area under the (fpr, tpr) staircase via the trapezoid rule."""
    trapezoid = getattr(np, "trapezoid", None) or np.trapz  # numpy 2.x renamed trapz
    return float(trapezoid(tpr, fpr))


def auc_rank_statistic(y_true, scores):
    """AUC = P(score of random positive > score of random negative), ties = 1/2.

    Computed via rank sums (Mann-Whitney U), O(n log n).
    """
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    n_pos = int(y_true.sum())
    n_neg = int(y_true.size - n_pos)

    # midranks (average rank for ties), 1-based
    order = np.argsort(scores, kind="stable")
    ranks = np.empty(scores.size)
    sorted_scores = scores[order]
    i = 0
    while i < scores.size:
        j = i
        while j + 1 < scores.size and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0  # average of ranks i+1..j+1
        i = j + 1

    rank_sum_pos = ranks[y_true == 1].sum()
    u = rank_sum_pos - n_pos * (n_pos + 1) / 2.0
    return float(u / (n_pos * n_neg))


# The 6-example table from notes.md section 2.
scores = np.array([0.95, 0.80, 0.70, 0.60, 0.40, 0.20])
labels = np.array([1, 1, 0, 1, 0, 0])
fpr, tpr, thr = roc_curve(labels, scores)
print(f"{'thresh':>8} {'FPR':>6} {'TPR':>6}")
for t, f, tp_rate in zip(thr, fpr, tpr):
    print(f"{t:>8.2f} {f:>6.3f} {tp_rate:>6.3f}")
a1 = auc_trapezoid(fpr, tpr)
a2 = auc_rank_statistic(labels, scores)
print(f"AUC (trapezoid over sweep) : {a1:.6f}")
print(f"AUC (rank statistic)       : {a2:.6f}   (expected 8/9 = {8/9:.6f})")
assert abs(a1 - a2) < 1e-12 and abs(a1 - 8 / 9) < 1e-12

try:
    from sklearn import metrics as skm
    sk_auc_hand = skm.roc_auc_score(labels, scores)
    print(f"AUC (sklearn)              : {sk_auc_hand:.6f}")
    assert abs(sk_auc_hand - a1) < 1e-12
except ImportError:
    print("(sklearn not installed; skipping the sklearn check)")

  thresh    FPR    TPR
     inf  0.000  0.000
    0.95  0.000  0.333
    0.80  0.000  0.667
    0.70  0.333  0.667
    0.60  0.333  1.000
    0.40  0.667  1.000
    0.20  1.000  1.000
AUC (trapezoid over sweep) : 0.888889
AUC (rank statistic)       : 0.888889   (expected 8/9 = 0.888889)


AUC (sklearn)              : 0.888889


## 3. PR curve, average precision, and a check against sklearn

`pr_curve` runs the same descending-threshold sweep but records (recall, precision), and
`average_precision` sums precision times the recall increase at each step. The cell
draws 500 positives with scores from N(1.5, 1) and 500 negatives from N(0, 1), computes
every metric so far (a hard threshold of 0.75 for precision, recall and F1), and
compares each one with sklearn. Expect the two AUC computations to agree with each other
and every value to match sklearn to within 1e-10.

In [4]:
def pr_curve(y_true, scores):
    """Returns (precision, recall) over the descending-threshold sweep."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="stable")
    y_sorted = y_true[order]
    s_sorted = scores[order]

    tps = np.cumsum(y_sorted)
    preds = np.arange(1, y_sorted.size + 1)

    distinct = np.where(np.diff(s_sorted))[0]
    idx = np.r_[distinct, y_sorted.size - 1]

    precision = tps[idx] / preds[idx]
    recall = tps[idx] / max(int(y_true.sum()), 1)
    return precision, recall


def average_precision(y_true, scores):
    """AP = sum over recall steps of precision, i.e. sum_k (R_k - R_{k-1}) P_k."""
    precision, recall = pr_curve(y_true, scores)
    recall = np.r_[0.0, recall]
    return float(np.sum(np.diff(recall) * precision))


def make_scores(n_pos, n_neg, rng):
    """Synthetic scores: positives ~ N(1.5, 1), negatives ~ N(0, 1).

    The per-class score distributions do not depend on the class ratio, so the
    ROC curve and AUC are the same in expectation; only the mix changes.
    """
    s_pos = rng.normal(1.5, 1.0, n_pos)
    s_neg = rng.normal(0.0, 1.0, n_neg)
    scores = np.r_[s_pos, s_neg]
    labels = np.r_[np.ones(n_pos, int), np.zeros(n_neg, int)]
    return labels, scores


rng = np.random.default_rng(42)
labels, scores = make_scores(500, 500, rng)
preds = (scores >= 0.75).astype(int)

tp, fp, fn, tn = confusion_matrix(labels, preds)
p, r, f1 = precision_score(labels, preds), recall_score(labels, preds), f1_score(labels, preds)
fpr, tpr, _ = roc_curve(labels, scores)
a_sweep = auc_trapezoid(fpr, tpr)
a_rank = auc_rank_statistic(labels, scores)
ap = average_precision(labels, scores)

print(f"confusion: TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"precision={p:.6f}  recall={r:.6f}  F1={f1:.6f}")
print(f"AUC sweep={a_sweep:.6f}  AUC rank={a_rank:.6f}  (diff={abs(a_sweep-a_rank):.2e})")
print(f"average precision={ap:.6f}")
assert abs(a_sweep - a_rank) < 1e-10, "two AUC computations must agree"

try:
    from sklearn import metrics as skm
except ImportError:
    skm = None
    print("(sklearn not installed; skipping verification)")
if skm is not None:
    sk_p = skm.precision_score(labels, preds)
    sk_r = skm.recall_score(labels, preds)
    sk_f = skm.f1_score(labels, preds)
    sk_auc = skm.roc_auc_score(labels, scores)
    sk_ap = skm.average_precision_score(labels, scores)
    print("sklearn check:")
    print(f"  precision {sk_p:.6f}  recall {sk_r:.6f}  F1 {sk_f:.6f}  AUC {sk_auc:.6f}  AP {sk_ap:.6f}")
    assert abs(p - sk_p) < 1e-10
    assert abs(r - sk_r) < 1e-10
    assert abs(f1 - sk_f) < 1e-10
    assert abs(a_rank - sk_auc) < 1e-10
    assert abs(ap - sk_ap) < 1e-10
    print("  all match to 1e-10.")

confusion: TP=386 FP=103 FN=114 TN=397
precision=0.789366  recall=0.772000  F1=0.780586
AUC sweep=0.864076  AUC rank=0.864076  (diff=0.00e+00)
average precision=0.861227
sklearn check:
  precision 0.789366  recall 0.772000  F1 0.780586  AUC 0.864076  AP 0.861227
  all match to 1e-10.


## 4. Imbalance: AUC stays constant, precision and AP drop

The score distributions stay fixed (positives N(1.5, 1), negatives N(0, 1)) while the
number of negatives per positive goes from 1 to 10 to 100. TPR and FPR are each computed
within one class, so AUC should stay near the same value (about 0.85) at every ratio. Precision at a fixed
threshold of 1.0, and AP, should fall steeply, because
precision = pi*TPR / (pi*TPR + (1-pi)*FPR) and the (1-pi)*FPR term grows with the number
of negatives.

In [5]:
rng = np.random.default_rng(42)
n_pos = 2000
header = f"{'neg:pos':>10} {'prevalence':>11} {'AUC':>8} {'AP':>8} {'prec@t=1':>9} {'rec@t=1':>8}"
print(header)
print("-" * len(header))
rows = []
for ratio in [1, 10, 100]:
    labels, scores = make_scores(n_pos, n_pos * ratio, rng)
    fpr, tpr, _ = roc_curve(labels, scores)
    auc = auc_trapezoid(fpr, tpr)
    ap = average_precision(labels, scores)
    preds = (scores >= 1.0).astype(int)
    p = precision_score(labels, preds)
    r = recall_score(labels, preds)
    prev = n_pos / (n_pos * (1 + ratio))
    rows.append((ratio, auc, ap, p))
    print(f"{ratio:>9}:1 {prev:>11.4f} {auc:>8.4f} {ap:>8.4f} {p:>9.4f} {r:>8.4f}")

aucs = [row[1] for row in rows]
aps = [row[2] for row in rows]
print()
print(f"AUC range across ratios: {max(aucs) - min(aucs):.4f}")
print(f"AP from 1:1 to 100:1: {aps[0]:.4f} -> {aps[-1]:.4f}")
assert max(aucs) - min(aucs) < 0.02, "AUC should not depend on the class ratio"
assert aps[0] > aps[1] > aps[2], "AP should fall as negatives are added"

   neg:pos  prevalence      AUC       AP  prec@t=1  rec@t=1
-----------------------------------------------------------
        1:1      0.5000   0.8440   0.8411    0.8121   0.6830
       10:1      0.0909   0.8552   0.4581    0.2959   0.6975
      100:1      0.0099   0.8596   0.1210    0.0424   0.6980

AUC range across ratios: 0.0156
AP from 1:1 to 100:1: 0.8411 -> 0.1210
